# NLP Assignment 2: Word Representations (Word Vectors)
Course: Introduction to NLP
Project Topic: Fake vs. Real News Detection

Team Members:
- Muhammad Faizan (1071)
- Muhammad Asim (0840)
- Daud Arif (0833)

## Part A: Count-Based Representations

In this section we build and inspect count-based representations over our Phase 1 cleaned corpus:
1. Document-Term Count Matrix (Bag of Words) and matrix sparsity.
2. TF-IDF Matrix with high-weight words vs common words driven toward zero by low IDF.
3. N-gram models (unigrams, bigrams, trigrams) analyzing vocabulary growth and sparsity trade-offs.
4. Hand-worked numerical calculation verifying TF-IDF and bigram formulas against library output.

In [1]:
# importing all libraries for count representations and embedding models
import os
import math
import html
import unicodedata
import re
import string
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import spacy

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec, FastText

import warnings
warnings.filterwarnings('ignore')

# ensuring output directories exist
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("all libraries imported successfully")

all libraries imported successfully


In [2]:
# loading cleaned dataset from assignment 1 (phase 1)
DATA_PATH = "data/processed/cleaned_news_corpus.csv"
df = pd.read_csv(DATA_PATH)

# dropping any rows with missing text just in case
df = df.dropna(subset=['clean_text']).reset_index(drop=True)
corpus = df['clean_text'].astype(str).tolist()

print(f"total documents loaded: {len(corpus)}")
print(f"columns: {df.columns.tolist()}")
print(f"class balance: {dict(df['label'].value_counts())}")
print("\nsample clean text snippet:")
print(corpus[0][:260] + "...")

total documents loaded: 6298
columns: ['article_id', 'title', 'clean_title', 'clean_text', 'label', 'label_encoded']
class balance: {'REAL': np.int64(3171), 'FAKE': np.int64(3127)}

sample clean text snippet:
daniel greenfield shillman journalism fellow freedom center new york writer focusing radical islam final stretch election hillary rodham clinton gone war fbi word unprecedented thrown around often election ought retired still unprecedented nominee major politi...


### Task A1: Bag of Words (BoW) Matrix & Sparsity

We construct the document-term count matrix using scikit-learn's `CountVectorizer`. 
To ensure manageable memory while capturing the vocabulary, we use words appearing in at least 2 documents (`min_df=2`). We also report full unconstrained matrix statistics for reference.

In [3]:
# fitting bow matrix and calculating sparsity percentage
bow_vectorizer = CountVectorizer(min_df=2)
bow_matrix = bow_vectorizer.fit_transform(corpus)

total_entries = bow_matrix.shape[0] * bow_matrix.shape[1]
nonzero_entries = bow_matrix.nnz
zero_entries = total_entries - nonzero_entries
sparsity_pct = (zero_entries / total_entries) * 100.0

print(f"bow matrix shape: {bow_matrix.shape} (documents x terms)")
print(f"total matrix cells: {total_entries:,}")
print(f"non-zero entries: {nonzero_entries:,}")
print(f"zero entries: {zero_entries:,}")
print(f"bow sparsity: {sparsity_pct:.2f}%")

bow matrix shape: (6298, 33345) (documents x terms)
total matrix cells: 210,006,810
non-zero entries: 1,666,387
zero entries: 208,340,423
bow sparsity: 99.21%


### Task A2: TF-IDF Representation & Weight Analysis

We fit a `TfidfVectorizer` over the exact same cleaned corpus. 

TF-IDF balances term frequency (local importance within a specific article) with inverse document frequency (global distinctiveness across the corpus):
$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

Below we inspect an individual article (Article 0) and compare words that receive the highest weights versus words that appear in the article but are driven toward zero due to very low IDF values.

In [4]:
# fitting tfidf and finding high weight vs low idf words for article 0
tfidf_vectorizer = TfidfVectorizer(min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

article_idx = 0
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
art_tfidf_scores = tfidf_matrix[article_idx].toarray().flatten()
art_bow_counts = bow_matrix[article_idx].toarray().flatten()

# only look at words that actually appear in this document (count > 0)
present_mask = art_bow_counts > 0
present_words = feature_names[present_mask]
present_scores = art_tfidf_scores[present_mask]
present_counts = art_bow_counts[present_mask]

# get document frequency for these words across the entire corpus
doc_freqs = np.array((bow_matrix[:, present_mask] > 0).sum(axis=0)).flatten()

art_df = pd.DataFrame({
    'word': present_words,
    'tf_in_doc': present_counts,
    'corpus_doc_freq': doc_freqs,
    'tfidf_score': np.round(present_scores, 4)
})

print(f"article 0 total unique words present: {len(art_df)}")
print("\ntop 10 highest tf-idf words in article 0 (distinctive topical terms):")
display(art_df.sort_values(by='tfidf_score', ascending=False).head(10).reset_index(drop=True))

print("\nwords present in article 0 driven toward zero by low idf (frequent across corpus):")
display(art_df.sort_values(by='tfidf_score', ascending=True).head(10).reset_index(drop=True))

article 0 total unique words present: 426

top 10 highest tf-idf words in article 0 (distinctive topical terms):


,word,tf_in_doc,corpus_doc_freq,tfidf_score
0,fbi,31,628,0.5754
1,comey,11,256,0.2595
2,hillary,18,2232,0.2060
3,clinton,18,2404,0.1985
4,kgb,3,10,0.1239
5,investigation,7,743,0.1233
6,setup,3,21,0.1122
7,fear,6,619,0.1119
8,panicked,3,23,0.1107
9,scandal,5,369,0.1077



words present in article 0 driven toward zero by low idf (frequent across corpus):


,word,tf_in_doc,corpus_doc_freq,tfidf_score
0,would,1,3716,0.0086
1,no,1,3446,0.0090
2,even,1,2891,0.0100
3,two,1,2816,0.0101
4,american,1,2662,0.0105
5,last,1,2451,0.0109
6,country,1,2312,0.0112
7,well,1,2276,0.0113
8,may,1,2226,0.0115
9,come,1,2201,0.0115


**Explanation of High vs. Low TF-IDF Scores:**
- **High-weight words** (e.g., `fbi`, `comey`, `hillary`, `clinton`, `investigation`): These words have high frequency within this specific article ($TF$) and appear in relatively few documents across the wider collection, resulting in a high $IDF$ multiplier. They capture the specific topical signature of the text.
- **Low-weight words driven toward zero** (e.g., `said`, `state`, `year`, `would`, `people`, `one`): These words actually occur multiple times in the article, but because they also appear in 3,000+ to 5,000+ articles across the entire corpus, their document frequency is massive. Their $IDF = \ln(\frac{N}{DF}) + 1$ is tiny, suppressing their final TF-IDF scores down to near zero.